# RT Notebook 26 — D/E Clean-Room Candidate Comparison

This single notebook:

- embeds the independent formal-oracle capture,
- implements a clean-room oracle,
- provides a marked candidate adapter,
- generates a new frozen corpus,
- compares representability, non-collapse, admissibility, and rejection codes,
- performs shuffled replay,
- preserves every disagreement,
- writes evidence artifacts and a ZIP archive.

The default candidate adapter is a **reference stand-in**. Replace only that marked cell with the exact governed P127/P128 implementation and set its mode to `GOVERNED_FROZEN_CANDIDATE` for an authoritative comparison.


In [ ]:

from pathlib import Path
import json, hashlib, random, copy, zipfile, platform, sys
from datetime import datetime, timezone

ROOT = Path.cwd()
NOTEBOOK_ID = "RT_NOTEBOOK_26_D_E_CLEANROOM_CANDIDATE_COMPARISON_001"
RESULT_ID = NOTEBOOK_ID + "_RESULTS_001"
OUT = ROOT / RESULT_ID
OUT.mkdir(parents=True, exist_ok=True)

def canonical(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False)

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def write_json(path, obj):
    path.write_text(json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False) + "\n", encoding="utf-8")

def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(canonical(row) + "\n")

print("Output:", OUT.resolve())


In [ ]:
FORMAL_ORACLE_SPEC = {'artifact_id': 'RT_D_E_INDEPENDENT_FORMAL_ORACLE_SPEC_20260731_001', 'artifact_type': 'INDEPENDENT_FORMAL_ORACLE_SPECIFICATION', 'authority_and_governance': {'authority_level': 'HUMAN_REVIEW_REQUIRED', 'canonical_promotion_allowed': False, 'claim_ceiling': 'C2_LIMITATION_OR_NEGATIVE_RESULT', 'mutation_policy': 'IMMUTABLE_AFTER_FREEZE', 'proof_status': 'NOT_ATTEMPTED', 'repository_action': 'ADD_SPECIFICATION_ONLY', 'required_reviewers': ['D-semantics reviewer', 'formal methods reviewer', 'governance reviewer']}, 'clean_room_implementation_protocol': {'acceptance_condition': 'The implementation is accepted as clean-room only after reviewer confirmation that no excluded source was consulted.', 'implementer_exclusions': ['candidate source code', 'Notebook 23 code', 'Notebook 24 code', 'Notebook 25 subject or oracle code', 'prior row-level expected labels'], 'implementer_inputs': ['this specification', 'a frozen threshold environment', 'a generated test corpus'], 'required_outputs': ['oracle evaluator source', 'row-level outputs', 'normalized output digest', 'implementation provenance', 'dependency lock or environment manifest']}, 'comparison_protocol': {'comparison_units': ['Representable_D result', 'NonCollapsed_E result', 'Admissible_DE result', 'rejection category', 'normalized replay digest'], 'oracle_under_test': 'clean-room implementation derived solely from this specification', 'preservation_rule': 'Every disagreement must be retained as a counterexample or unresolved semantic ambiguity.', 'required_case_families': ['valid baselines', 'single-fault mutations', 'combined faults', 'threshold boundary triples', 'permutation replay', 'nonsemantic enrichment', 'adversarial countermodels'], 'subject_under_test': 'P127_P128_CANDIDATE_20260730_001 or governed successor'}, 'content_sha256': '7c91f38595d411262926a365ca8a26d997bcf96bff4335bf64ed25bd174ac025', 'countermodel_classes': [{'class': 'TYPE_CONFUSION', 'construction': 'Use a relation_type outside the singleton allowed domain.', 'id': 'CM-DE-001'}, {'class': 'CONTEXT_LEAKAGE', 'construction': 'Bind source and target to distinct context identifiers.', 'id': 'CM-DE-002'}, {'class': 'WITNESS_ALIASING', 'construction': 'Use a witness token with equal members but altered order or structure.', 'id': 'CM-DE-003'}, {'class': 'HISTORY_REORDERING', 'construction': 'Reverse or duplicate history step indices.', 'id': 'CM-DE-004'}, {'class': 'FALSE_TERMINAL', 'construction': 'Use a valid ordered history whose final state differs from target.', 'id': 'CM-DE-005'}, {'class': 'THRESHOLD_EDGE', 'construction': 'Test distinction values epsilon below, exactly at, and epsilon above the threshold.', 'id': 'CM-DE-006'}, {'class': 'UNKNOWN_PROFILE', 'construction': 'Use a profile absent from the threshold environment.', 'id': 'CM-DE-007'}, {'class': 'COMBINED_FAULT', 'construction': 'Combine multiple invalid conditions to test rejection precedence.', 'id': 'CM-DE-008'}], 'created_at': '2026-07-31T00:00:00Z', 'decision_rules': {'FAIL_COUNTEREXAMPLE': 'Any preserved in-domain disagreement, nondeterministic result, independence breach, or specification mismatch is found.', 'PARTIAL_LIMITATION': 'Execution completes but independence, coverage, provenance, or semantic ambiguity remains unresolved.', 'PASS_BOUNDED': 'No disagreement is found within the frozen declared domain, clean-room conditions are verified, and replay is deterministic.'}, 'domain': ['RT', 'D-semantics', 'E-semantics', 'formal oracle', 'bounded semantic verification'], 'expected_evidence_classes': {'bounded_disagreement': 'C2_LIMITATION_OR_NEGATIVE_RESULT', 'clean_computational_agreement': 'C2_BOUNDED_SUPPORT', 'formal_equivalence_proof': 'C3_CANDIDATE_FORMAL_SUPPORT_PENDING_GOVERNANCE', 'universal_theorem': 'NOT_AUTHORIZED_BY_THIS_SPEC'}, 'explicit_non_authorizations': ['No claim promotion above C2.', 'No theorem promotion.', 'No D-obligation discharge.', 'No textbook semantic replacement.', 'No registry closure.', 'No suppression of counterexamples.', 'No inference of universal correctness.'], 'falsification_conditions': ['The clean-room oracle disagrees with the formal definitions in this specification.', 'The candidate implementation disagrees with the clean-room oracle on any in-domain case.', 'Rejection precedence is ambiguous or implementation-dependent.', 'Evaluation order changes normalized outputs.', 'Unknown profiles are admitted.', 'Threshold equality is admitted as non-collapsed.', 'Witness or history violations are accepted.', 'A reviewer finds that the oracle reused candidate implementation logic.', 'The canonical artifact hash or frozen specification changes after review begins.'], 'formal_relations': {'Admissible_DE': {'definition': 'Admissible_DE(r, env) := Representable_D(r, env) ∧ NonCollapsed_E(r, env)', 'signature': 'DRecord x ThresholdEnvironment -> Bool'}, 'NonCollapsed_E': {'definition_conjunction': ['KnownProfile(record.profile, env)', 'PositiveDistinction(record.distinction)', 'AboveThreshold(record.distinction, record.profile, env)'], 'formal_schema': 'NonCollapsed_E(r, env) := KnownProfile(r.profile, env) ∧ PositiveDistinction(r.distinction) ∧ r.distinction > env[r.profile]', 'signature': 'DRecord x ThresholdEnvironment -> Bool'}, 'Representable_D': {'definition_conjunction': ['record.relation_type = SourceRelation', 'SameContext(record.context, record.target_context)', 'KnownProfile(record.profile, env)', 'record.witness has type Witness', 'ExactBind(record.witness, record.source_payload)', 'record.history has type History', 'OrderedHistory(record.history)', 'TerminatesAt(record.history, record.target)'], 'formal_schema': 'Representable_D(r, env) := TypeOK(r) ∧ SameContext(r.context, r.target_context) ∧ KnownProfile(r.profile, env) ∧ WitnessShapeOK(r.witness) ∧ ExactBind(r.witness, r.source_payload) ∧ HistoryShapeOK(r.history) ∧ OrderedHistory(r.history) ∧ TerminatesAt(r.history, r.target)', 'signature': 'DRecord x ThresholdEnvironment -> Bool'}}, 'independence_requirements': {'independence_test': {'condition': 'A clean-room implementer must be able to construct the oracle from this specification without reading the candidate implementation.', 'failure_condition': 'Any oracle rule is justified only by reference to implementation behavior rather than declared semantics.'}, 'must_be_derived_from': ['declared D/E semantic relations', 'typed domain definitions', 'explicit admissibility conditions', 'explicit rejection precedence'], 'must_not_call': ['candidate predicate functions', 'subject implementation helpers', 'prior expected-label fixtures'], 'must_not_import': ['P127 implementation', 'P128 implementation', 'Notebook 23 evaluator', 'Notebook 24 evaluator', 'Notebook 25 subject functions'], 'permitted_reuse': ['symbol names', 'registry object identifiers', 'domain vocabulary', 'non-executable provenance metadata']}, 'proof_obligations': [{'id': 'PO-DE-ORACLE-001', 'statement': 'Prove that the clean-room evaluator is extensionally equivalent to Representable_D.'}, {'id': 'PO-DE-ORACLE-002', 'statement': 'Prove that the clean-room evaluator is extensionally equivalent to NonCollapsed_E.'}, {'id': 'PO-DE-ORACLE-003', 'statement': 'Prove rejection precedence is total and deterministic for all in-domain records.'}, {'id': 'PO-DE-ORACLE-004', 'statement': 'Prove nonsemantic enrichment invariance.'}, {'id': 'PO-DE-ORACLE-005', 'statement': 'Prove evaluation-order independence.'}, {'id': 'PO-DE-ORACLE-006', 'statement': 'Determine whether Admissible_DE preserves the intended D/E distinction without collapse.'}], 'provenance': {'generation_note': 'Specification authored as an additive independent formalization artifact. It does not certify any existing implementation.', 'source_artifacts': ['MPF_SIM_D_E_METAMORPHIC_INDEPENDENCE_001_RESULTS_001', 'MPF_SIM_D_E_ORACLE_ADVERSARIAL_001_RESULTS_001', 'MPF_SIM_D_E_ORACLE_ADVERSARIAL_001_PROCESSING_001'], 'source_context': ['Notebook 24 metamorphic independence review', 'Notebook 25 oracle adversarial execution', 'approved-tool replication requirement', 'oracle-independence frontier']}, 'purpose': {'explicit_nonpurpose': ['This artifact does not prove universal D/E preservation.', 'This artifact does not promote any theorem.', 'This artifact does not raise the claim ceiling above C2.', 'This artifact does not authorize closure of the D-semantics program.'], 'primary': 'Define an implementation-independent formal oracle for evaluating bounded D/E admissibility and non-collapse claims.', 'secondary': ['Provide a semantic reference that does not import or call the candidate implementation.', 'Expose hidden shared assumptions between the candidate predicates and prior declarative oracles.', 'Support later Lean formalization, independent reimplementation, and adversarial comparison.']}, 'recommended_next_action': {'action': 'IMPLEMENT_CLEAN_ROOM_ORACLE', 'preferred_target': 'Lean 4 formalization plus separately authored Python reference evaluator', 'sequence': ['Human review and freeze this specification.', 'Assign a clean-room implementer.', 'Implement the relations without candidate-code access.', 'Generate a new adversarial corpus.', 'Compare candidate and oracle outputs.', 'Preserve all disagreements.', 'Perform bounded governed review.']}, 'rejection_precedence': {'NonCollapsed_E': [{'condition': 'profile not in threshold environment', 'rank': 1, 'result': 'REJECT_PROFILE'}, {'condition': 'distinction is nonnumeric, nonfinite, or <= 0', 'rank': 2, 'result': 'REJECT_DISTINCTION'}, {'condition': 'distinction <= threshold(profile)', 'rank': 3, 'result': 'REJECT_SUBTHRESHOLD'}, {'condition': 'distinction > threshold(profile)', 'rank': 4, 'result': 'NON_COLLAPSED'}], 'Representable_D': [{'condition': 'relation_type != SourceRelation', 'rank': 1, 'result': 'REJECT_TYPE'}, {'condition': 'context != target_context', 'rank': 2, 'result': 'REJECT_CONTEXT'}, {'condition': 'profile not in threshold environment', 'rank': 3, 'result': 'REJECT_PROFILE'}, {'condition': 'witness is absent or malformed', 'rank': 4, 'result': 'REJECT_WITNESS'}, {'condition': 'witness.token is not exactly equal to source_payload', 'rank': 5, 'result': 'REJECT_WITNESS'}, {'condition': 'history is absent, malformed, or empty', 'rank': 6, 'result': 'REJECT_HISTORY'}, {'condition': 'history step order is not strictly increasing', 'rank': 7, 'result': 'REJECT_HISTORY'}, {'condition': 'history does not terminate at target', 'rank': 8, 'result': 'REJECT_HISTORY'}, {'condition': 'all prior conditions pass', 'rank': 9, 'result': 'REPRESENTABLE'}]}, 'required_deliverables': ['frozen_specification.json', 'clean_room_oracle_source', 'test_corpus.jsonl', 'oracle_outputs.jsonl', 'candidate_outputs.jsonl', 'comparison_report.json', 'falsification_report.json', 'manifest.json', 'archive.zip'], 'schema_version': '1.0.0', 'semantic_invariants': [{'falsifier': 'Representable_D remains true when context != target_context.', 'id': 'INV-DE-001', 'name': 'Context non-transport', 'statement': 'Changing target_context while holding context fixed must not preserve Representable_D unless equality is restored.'}, {'falsifier': 'Representable_D remains true with a non-equal witness token.', 'id': 'INV-DE-002', 'name': 'Exact witness binding', 'statement': 'Any structural change to witness.token that breaks canonical equality with source_payload must make Representable_D false.'}, {'falsifier': 'Representable_D remains true for unordered history.', 'id': 'INV-DE-003', 'name': 'History order necessity', 'statement': 'A non-increasing or reordered history must make Representable_D false.'}, {'falsifier': 'Representable_D remains true when the history terminal differs from target.', 'id': 'INV-DE-004', 'name': 'History terminal necessity', 'statement': 'Changing the final history state away from target must make Representable_D false.'}, {'falsifier': 'Equality at threshold is admitted or a strictly greater positive distinction is rejected.', 'id': 'INV-DE-005', 'name': 'Threshold strictness', 'statement': 'NonCollapsed_E is false at and below threshold and true only strictly above threshold.'}, {'falsifier': 'Either relation admits an unknown profile.', 'id': 'INV-DE-006', 'name': 'Unknown profile rejection', 'statement': 'Both Representable_D and NonCollapsed_E must reject profiles absent from the threshold environment.'}, {'falsifier': 'Outputs vary solely because of undeclared nonsemantic metadata.', 'id': 'INV-DE-007', 'name': 'Nonsemantic enrichment invariance', 'statement': 'Adding fields not referenced by the formal relations must not change semantic output.'}, {'falsifier': 'Normalized outputs differ under row-order permutation.', 'id': 'INV-DE-008', 'name': 'Evaluation-order independence', 'statement': 'Permutation of the evaluation order must not alter normalized outputs.'}], 'semantic_primitives': {'AboveThreshold': {'definition': 'AboveThreshold(d, p, env) is true iff KnownProfile(p, env) and d > env[p].', 'signature': 'DistinctionValue x ProfileId x ThresholdEnvironment -> Bool'}, 'ExactBind': {'definition': 'ExactBind(w, p) is true iff w.token is structurally equal to p under canonical ordered equality.', 'signature': 'Witness x Payload -> Bool'}, 'KnownProfile': {'definition': 'KnownProfile(p, env) is true iff p is in the domain of env.', 'signature': 'ProfileId x ThresholdEnvironment -> Bool'}, 'OrderedHistory': {'definition': 'OrderedHistory(h) is true iff h is nonempty and its step indices are strictly increasing.', 'signature': 'History -> Bool'}, 'PositiveDistinction': {'definition': 'PositiveDistinction(d) is true iff d > 0.', 'signature': 'DistinctionValue -> Bool'}, 'SameContext': {'definition': 'SameContext(c1, c2) is true iff c1 and c2 are identical opaque context identifiers.', 'signature': 'ContextId x ContextId -> Bool'}, 'TerminatesAt': {'definition': 'TerminatesAt(h, t) is true iff the final history element has state equal to t.', 'signature': 'History x State -> Bool'}}, 'status': 'PROPOSED_FROZEN_SPEC', 'typed_domain': {'types': {'ContextId': {'constraint': 'non-empty string', 'kind': 'opaque_identifier'}, 'DRecord': {'kind': 'record', 'required_fields': ['relation_type', 'context', 'target_context', 'source_payload', 'witness', 'history', 'target', 'profile', 'distinction']}, 'DistinctionValue': {'constraint': 'finite real-valued scalar in the bounded computational model', 'kind': 'ordered_numeric'}, 'History': {'element_type': 'HistoryStep', 'kind': 'finite_nonempty_ordered_sequence'}, 'HistoryStep': {'kind': 'record', 'required_fields': ['step', 'state']}, 'Payload': {'constraint': 'must support exact equality', 'kind': 'finite_ordered_structure'}, 'ProfileId': {'constraint': 'must resolve through an explicit threshold environment', 'kind': 'opaque_identifier'}, 'RelationType': {'allowed': ['SourceRelation'], 'kind': 'enumeration'}, 'ThresholdEnvironment': {'kind': 'finite_mapping', 'maps': 'ProfileId -> positive DistinctionValue'}, 'Witness': {'kind': 'record', 'required_fields': ['token']}}}}
SPEC_CAPTURE = {
    "artifact_id": FORMAL_ORACLE_SPEC.get("artifact_id"),
    "content_sha256": FORMAL_ORACLE_SPEC.get("content_sha256"),
    "captured_for": NOTEBOOK_ID
}
write_json(OUT / "formal_oracle_spec_capture.json", SPEC_CAPTURE)
SPEC_CAPTURE


In [ ]:

THRESHOLD_ENV = {"alpha": 0.125, "beta": 0.375, "gamma": 0.625, "delta": 0.875}
CONTEXTS = ["K1", "K2", "K3", "K4"]
SEEDS = [2601, 2609, 2617, 2621]
write_json(OUT / "threshold_environment.json", THRESHOLD_ENV)
THRESHOLD_ENV


## Clean-room oracle

In [ ]:

def cr_same_context(a, b):
    return isinstance(a, str) and isinstance(b, str) and len(a) > 0 and a == b

def cr_exact_bind(witness, payload):
    return isinstance(witness, dict) and witness.get("token") == payload

def cr_ordered_history(history):
    if not isinstance(history, list) or not history:
        return False
    if not all(isinstance(x, dict) and "step" in x and "state" in x for x in history):
        return False
    steps = [x["step"] for x in history]
    if not all(isinstance(s, int) and not isinstance(s, bool) for s in steps):
        return False
    return all(steps[i] < steps[i + 1] for i in range(len(steps) - 1))

def cleanroom_representable(r, env):
    checks = [
        ("REJECT_TYPE", r.get("relation_type") == "SourceRelation"),
        ("REJECT_CONTEXT", cr_same_context(r.get("context"), r.get("target_context"))),
        ("REJECT_PROFILE", r.get("profile") in env),
        ("REJECT_WITNESS", isinstance(r.get("witness"), dict)),
        ("REJECT_WITNESS", cr_exact_bind(r.get("witness"), r.get("source_payload"))),
        ("REJECT_HISTORY", isinstance(r.get("history"), list) and len(r.get("history")) > 0),
        ("REJECT_HISTORY", cr_ordered_history(r.get("history"))),
        ("REJECT_HISTORY", isinstance(r.get("history"), list) and len(r.get("history")) > 0 and r["history"][-1].get("state") == r.get("target"))
    ]
    for rejection, passed in checks:
        if not passed:
            return rejection
    return "REPRESENTABLE"

def cleanroom_noncollapsed(r, env):
    profile = r.get("profile")
    if profile not in env:
        return "REJECT_PROFILE"
    d = r.get("distinction")
    if not isinstance(d, (int, float)) or isinstance(d, bool) or d <= 0:
        return "REJECT_DISTINCTION"
    if d <= env[profile]:
        return "REJECT_SUBTHRESHOLD"
    return "NON_COLLAPSED"

def cleanroom_admissible(r, env):
    return cleanroom_representable(r, env) == "REPRESENTABLE" and cleanroom_noncollapsed(r, env) == "NON_COLLAPSED"


## Candidate adapter

Replace this cell with the exact governed frozen candidate implementation for the authoritative run.


In [ ]:

CANDIDATE_ADAPTER_MODE = "REFERENCE_STANDIN"
CANDIDATE_IMPLEMENTATION_ID = "REFERENCE_CANDIDATE_ADAPTER_20260731_001"

def candidate_representable(r, env):
    if r.get("relation_type") != "SourceRelation":
        return "REJECT_TYPE"
    if r.get("context") != r.get("target_context"):
        return "REJECT_CONTEXT"
    if r.get("profile") not in env:
        return "REJECT_PROFILE"
    witness = r.get("witness")
    if not isinstance(witness, dict) or witness.get("token") != r.get("source_payload"):
        return "REJECT_WITNESS"
    history = r.get("history")
    if not isinstance(history, list) or not history:
        return "REJECT_HISTORY"
    if not all(isinstance(x, dict) and "step" in x and "state" in x for x in history):
        return "REJECT_HISTORY"
    steps = [x["step"] for x in history]
    if not all(isinstance(s, int) and not isinstance(s, bool) for s in steps):
        return "REJECT_HISTORY"
    if any(steps[i] >= steps[i+1] for i in range(len(steps)-1)):
        return "REJECT_HISTORY"
    if history[-1].get("state") != r.get("target"):
        return "REJECT_HISTORY"
    return "REPRESENTABLE"

def candidate_noncollapsed(r, env):
    profile = r.get("profile")
    if profile not in env:
        return "REJECT_PROFILE"
    d = r.get("distinction")
    if not isinstance(d, (int, float)) or isinstance(d, bool) or d <= 0:
        return "REJECT_DISTINCTION"
    if d <= env[profile]:
        return "REJECT_SUBTHRESHOLD"
    return "NON_COLLAPSED"

def candidate_admissible(r, env):
    return candidate_representable(r, env) == "REPRESENTABLE" and candidate_noncollapsed(r, env) == "NON_COLLAPSED"


In [ ]:

def make_baseline(seed, context, profile, ordinal):
    rng = random.Random(f"N26:{seed}:{context}:{profile}:{ordinal}")
    payload = {"left": rng.randint(1, 1000), "right": [rng.randint(1, 50), rng.randint(51, 100)]}
    target = f"TARGET_{context}_{seed}_{ordinal}"
    threshold = THRESHOLD_ENV[profile]
    return {
        "row_id": f"N26_B_{seed}_{context}_{profile}_{ordinal}",
        "relation_type": "SourceRelation",
        "context": context,
        "target_context": context,
        "source_payload": payload,
        "witness": {"token": copy.deepcopy(payload), "review": "independent"},
        "history": [
            {"step": 10, "state": f"START_{ordinal}"},
            {"step": 20, "state": f"MID_{ordinal}"},
            {"step": 30, "state": target}
        ],
        "target": target,
        "profile": profile,
        "distinction": round(threshold + 0.01 + rng.random() * 0.05, 9),
        "metadata": {"seed": seed, "ordinal": ordinal},
        "family": "baseline"
    }

BASELINES = [
    make_baseline(seed, context, profile, ordinal)
    for seed in SEEDS
    for context in CONTEXTS
    for profile in THRESHOLD_ENV
    for ordinal in range(2)
]

FAULTS = [
    "wrong_type", "cross_context", "missing_witness", "witness_corruption",
    "empty_history", "duplicate_history_step", "descending_history", "false_terminal",
    "unknown_profile", "zero_distinction", "negative_distinction",
    "threshold_minus_epsilon", "threshold_exact", "threshold_plus_epsilon",
    "metadata_enrichment", "combined_fault_precedence"
]

def mutate(row, family):
    r = copy.deepcopy(row)
    r["row_id"] = f"{row['row_id']}__{family}"
    r["parent_id"] = row["row_id"]
    r["family"] = family
    if family == "wrong_type":
        r["relation_type"] = "OtherRelation"
    elif family == "cross_context":
        r["target_context"] = next(c for c in CONTEXTS if c != r["context"])
    elif family == "missing_witness":
        r["witness"] = None
    elif family == "witness_corruption":
        r["witness"]["token"]["left"] += 1
    elif family == "empty_history":
        r["history"] = []
    elif family == "duplicate_history_step":
        r["history"][1]["step"] = r["history"][0]["step"]
    elif family == "descending_history":
        r["history"] = list(reversed(r["history"]))
    elif family == "false_terminal":
        r["history"][-1]["state"] = "WRONG_TARGET"
    elif family == "unknown_profile":
        r["profile"] = "omega"
    elif family == "zero_distinction":
        r["distinction"] = 0
    elif family == "negative_distinction":
        r["distinction"] = -0.5
    elif family == "threshold_minus_epsilon":
        r["distinction"] = THRESHOLD_ENV[r["profile"]] - 1e-9
    elif family == "threshold_exact":
        r["distinction"] = THRESHOLD_ENV[r["profile"]]
    elif family == "threshold_plus_epsilon":
        r["distinction"] = THRESHOLD_ENV[r["profile"]] + 1e-9
    elif family == "metadata_enrichment":
        r["metadata"]["extra"] = {"nested": [1, 2, 3], "note": "nonsemantic"}
    elif family == "combined_fault_precedence":
        r["relation_type"] = "OtherRelation"
        r["target_context"] = "BAD_CONTEXT"
        r["profile"] = "omega"
        r["witness"] = None
        r["history"] = []
        r["distinction"] = -1
    return r

ROWS = []
for b in BASELINES:
    ROWS.append(copy.deepcopy(b))
    ROWS.extend(mutate(b, f) for f in FAULTS)

print("Baselines:", len(BASELINES))
print("Fault families:", len(FAULTS))
print("Rows per pass:", len(ROWS))


In [ ]:

def eval_cleanroom(row):
    return {
        "row_id": row["row_id"],
        "representable": cleanroom_representable(row, THRESHOLD_ENV),
        "noncollapsed": cleanroom_noncollapsed(row, THRESHOLD_ENV),
        "admissible": cleanroom_admissible(row, THRESHOLD_ENV)
    }

def eval_candidate(row):
    return {
        "row_id": row["row_id"],
        "representable": candidate_representable(row, THRESHOLD_ENV),
        "noncollapsed": candidate_noncollapsed(row, THRESHOLD_ENV),
        "admissible": candidate_admissible(row, THRESHOLD_ENV)
    }

def compare_row(row):
    candidate = eval_candidate(row)
    cleanroom = eval_cleanroom(row)
    mismatches = [k for k in ("representable", "noncollapsed", "admissible") if candidate[k] != cleanroom[k]]
    return {
        "row_id": row["row_id"],
        "parent_id": row.get("parent_id"),
        "family": row["family"],
        "candidate": candidate,
        "cleanroom": cleanroom,
        "agreement": len(mismatches) == 0,
        "mismatch_fields": mismatches,
        "input_sha256": sha256_bytes(canonical(row).encode("utf-8"))
    }

PASS1 = [compare_row(r) for r in ROWS]
rng = random.Random(26012601)
shuffled_rows = copy.deepcopy(ROWS)
rng.shuffle(shuffled_rows)
PASS2 = [compare_row(r) for r in shuffled_rows]

def normalized_digest(results):
    return sha256_bytes(canonical(sorted(results, key=lambda x: x["row_id"])).encode("utf-8"))

DIGEST1 = normalized_digest(PASS1)
DIGEST2 = normalized_digest(PASS2)
REPLAY_AGREEMENT = DIGEST1 == DIGEST2
COUNTEREXAMPLES = [x for x in PASS1 if not x["agreement"]]

print("Replay agreement:", REPLAY_AGREEMENT)
print("Counterexamples:", len(COUNTEREXAMPLES))


In [ ]:

RESULTS = {x["row_id"]: x for x in PASS1}
INVARIANTS = []

for b in BASELINES:
    bid = b["row_id"]
    INVARIANTS.append({
        "id": "baseline_admission",
        "row_id": bid,
        "pass": (
            RESULTS[bid]["cleanroom"]["representable"] == "REPRESENTABLE"
            and RESULTS[bid]["cleanroom"]["noncollapsed"] == "NON_COLLAPSED"
            and RESULTS[bid]["cleanroom"]["admissible"] is True
        )
    })
    for fam in ["threshold_minus_epsilon", "threshold_exact"]:
        rid = f"{bid}__{fam}"
        INVARIANTS.append({
            "id": "strict_threshold_rejection",
            "row_id": rid,
            "pass": RESULTS[rid]["cleanroom"]["noncollapsed"] == "REJECT_SUBTHRESHOLD"
        })
    rid = f"{bid}__threshold_plus_epsilon"
    INVARIANTS.append({
        "id": "strict_threshold_admission",
        "row_id": rid,
        "pass": RESULTS[rid]["cleanroom"]["noncollapsed"] == "NON_COLLAPSED"
    })
    rid = f"{bid}__combined_fault_precedence"
    INVARIANTS.append({
        "id": "combined_fault_precedence",
        "row_id": rid,
        "pass": RESULTS[rid]["cleanroom"]["representable"] == "REJECT_TYPE"
    })
    rid = f"{bid}__metadata_enrichment"
    INVARIANTS.append({
        "id": "metadata_enrichment_invariance",
        "row_id": rid,
        "pass": (
            RESULTS[rid]["cleanroom"]["representable"] == RESULTS[bid]["cleanroom"]["representable"]
            and RESULTS[rid]["cleanroom"]["noncollapsed"] == RESULTS[bid]["cleanroom"]["noncollapsed"]
        )
    })

INVARIANT_FAILURES = [x for x in INVARIANTS if not x["pass"]]
print("Invariant checks:", len(INVARIANTS))
print("Invariant failures:", len(INVARIANT_FAILURES))


In [ ]:

authoritative = CANDIDATE_ADAPTER_MODE == "GOVERNED_FROZEN_CANDIDATE"
if COUNTEREXAMPLES or INVARIANT_FAILURES or not REPLAY_AGREEMENT:
    outcome = "FAIL_OR_COUNTEREXAMPLE_PRESERVED"
elif authoritative:
    outcome = "PASS_BOUNDED_EQUIVALENCE"
else:
    outcome = "PASS_HARNESS_VALIDATION_REFERENCE_STANDIN"

summary = {
    "notebook_id": NOTEBOOK_ID,
    "result_id": RESULT_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "candidate_adapter_mode": CANDIDATE_ADAPTER_MODE,
    "candidate_implementation_id": CANDIDATE_IMPLEMENTATION_ID,
    "authoritative_governed_comparison": authoritative,
    "claim_ceiling": "C2_LIMITATION_OR_NEGATIVE_RESULT",
    "counts": {
        "baselines": len(BASELINES),
        "fault_families": len(FAULTS),
        "rows_per_pass": len(ROWS),
        "total_evaluations": len(ROWS) * 2,
        "counterexamples": len(COUNTEREXAMPLES),
        "invariant_checks": len(INVARIANTS),
        "invariant_failures": len(INVARIANT_FAILURES)
    },
    "replay": {"pass1_digest": DIGEST1, "pass2_digest": DIGEST2, "agreement": REPLAY_AGREEMENT},
    "outcome": outcome,
    "blocked_interpretations": [
        "Universal equivalence",
        "Formal proof",
        "Theorem promotion",
        "Claim promotion above C2",
        "Governed equivalence unless authoritative_governed_comparison is true"
    ]
}

falsification = {
    "result_id": RESULT_ID,
    "counterexamples": COUNTEREXAMPLES,
    "invariant_failures": INVARIANT_FAILURES,
    "replay_agreement": REPLAY_AGREEMENT,
    "preservation_rule": "All disagreements and invariant failures are retained without deletion."
}

write_jsonl(OUT / "comparison_corpus.jsonl", ROWS)
write_jsonl(OUT / "candidate_outputs.jsonl", [eval_candidate(r) for r in ROWS])
write_jsonl(OUT / "cleanroom_outputs.jsonl", [eval_cleanroom(r) for r in ROWS])
write_jsonl(OUT / "comparison_rows.jsonl", PASS1)
write_jsonl(OUT / "comparison_rows_replay_shuffled.jsonl", PASS2)
write_jsonl(OUT / "invariant_checks.jsonl", INVARIANTS)
write_json(OUT / "comparison_summary.json", summary)
write_json(OUT / "counterexamples.json", falsification)

summary


In [ ]:

artifact_files = sorted(
    p for p in OUT.iterdir()
    if p.is_file() and p.name not in {"manifest.json", f"{RESULT_ID}.zip"}
)

manifest = {
    "notebook_id": NOTEBOOK_ID,
    "result_id": RESULT_ID,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "candidate_adapter_mode": CANDIDATE_ADAPTER_MODE,
    "authoritative_governed_comparison": CANDIDATE_ADAPTER_MODE == "GOVERNED_FROZEN_CANDIDATE",
    "runtime": {"python": sys.version, "platform": platform.platform()},
    "artifacts": [
        {"name": p.name, "bytes": p.stat().st_size, "sha256": sha256_bytes(p.read_bytes())}
        for p in artifact_files
    ],
    "outcome": summary["outcome"],
    "claim_ceiling": summary["claim_ceiling"]
}
write_json(OUT / "manifest.json", manifest)

bundle = OUT / f"{RESULT_ID}.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p != bundle:
            z.write(p, arcname=p.name)

print("Archive:", bundle.resolve())
print("Archive SHA-256:", sha256_bytes(bundle.read_bytes()))


## Decision rule

`PASS_BOUNDED_EQUIVALENCE` is available only when the marked adapter contains the exact governed frozen candidate and its mode is `GOVERNED_FROZEN_CANDIDATE`.

The default included adapter produces `PASS_HARNESS_VALIDATION_REFERENCE_STANDIN`. That validates the notebook harness only.
